# MTI830 — Pipeline de sélection des joueurs et des parties

Ce notebook exécute la pipeline suivante :

1. Charger une liste manuelle de joueurs candidats depuis `players.txt`
2. Télécharger leurs parties brutes via l'API Lichess/Berserk
3. Sauvegarder le brut API (`raw_games_by_player.json`)
4. Construire `player_games_raw` via `build_player_games(...)`
5. Sauvegarder `player_games_raw.json`
6. Appliquer `select_player_games(...)`
7. Inspecter les résultats

Avant d'exécuter le notebook, vérifie que :

- `players.txt` existe à la racine du projet
- tes modules `src.ingestion.player_sampling`, `src.dataset.io`, `src.dataset.select_players` sont à jour
- ton token API Lichess est disponible


## 0. Imports et configuration


In [5]:
from pathlib import Path
import os
import sys
sys.path.append(os.path.abspath(".."))
import json
from dotenv import load_dotenv
import time
import datetime 
from datetime import timedelta

import pandas as pd
import berserk

from src.ingestion.player_sampling import collect_raw_games_for_players
from src.dataset.io import (
    save_raw_games_by_player,
    load_raw_games_by_player,
    build_player_games_raw_from_api_dump,
    save_player_games_raw,
    load_player_games_raw,
    save_raw_games_for_one_player,
    list_downloaded_player_ids,
    load_raw_games_by_player_from_directory,
    save_player_games_selected,
    load_player_games_selected
)
from src.dataset.select_players import PlayerSelectionConfig, select_player_games


In [18]:
# Ajuste ce chemin si nécessaire
PLAYERS_TXT_PATH = Path("../data/player_candidates/players.txt")
RAW_GAMES_JSON_PATH = Path("../data/raw/raw_games_by_player.json")
PLAYER_GAMES_RAW_JSON_PATH = Path("../data/raw/player_games_raw.json")
RAW_GAMES_BY_PLAYER_DIR = Path("../data/raw/by_player")
FAILED_PLAYERS_PATH = Path("../data/raw/failed_players.txt")
PLAYER_GAMES_SELECTED_JSON_PATH = Path("../data/selected/player_games_selected.json")
SELECTED_PLAYERS_TXT_PATH = Path("../data/selected/selected_players.txt")
SELECTION_SUMMARY_JSON_PATH = Path("../data/selected/selection_summary.json")
SELECTION_CONFIG_JSON_PATH = Path("../data/selected/selection_config.json")
GAMES_FINAL_JSON_PATH = Path("../data/enriched/games_final.json")
FINAL_DATASET_JSON_PATH = Path("../data/final/final_dataset.json")

# Pour un premier test, garde une limite raisonnable
MAX_GAMES_PER_PLAYER = 1021
SLEEP_SECONDS = 0.5

# Option pratique : recharger depuis disque au lieu de retélécharger
USE_EXISTING_RAW_GAMES_JSON = False
USE_EXISTING_PLAYER_GAMES_RAW_JSON = False

START_2025 = berserk.utils.to_millis(datetime.datetime(2025, 1, 1))
END_2026 = berserk.utils.to_millis(datetime.datetime(2026, 1, 1))


## 1. Création du client Berserk


In [13]:
# Option 1 : variable d'environnement LICHESS_API_TOKEN
load_dotenv("../.env")
token = os.getenv('LICHESS_API_TOKEN')

# Option 2 : décommente et colle temporairement ton token si besoin
# token = 'COLLE_ICI_TON_TOKEN'

if not token:
    raise ValueError('Token API introuvable. Définis LICHESS_API_TOKEN ou renseigne token dans la cellule.')

session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print('Client Berserk créé avec succès')


Client Berserk créé avec succès


## 2. Charger la liste manuelle de joueurs


In [14]:
def load_player_ids_from_txt(path: str | Path) -> list[str]:
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    return [line.strip() for line in lines if line.strip()]

seed_player_ids = load_player_ids_from_txt(PLAYERS_TXT_PATH)
print(f'{len(seed_player_ids)} joueurs chargés')
print(seed_player_ids[:15])

downloaded_player_ids = set(list_downloaded_player_ids(RAW_GAMES_BY_PLAYER_DIR))
remaining_player_ids = [p for p in seed_player_ids if p not in downloaded_player_ids]

print(f"Déjà téléchargés : {len(downloaded_player_ids)}")
print(f"Restants à télécharger : {len(remaining_player_ids)}")


1892 joueurs chargés
['VW-Leoo', 'Lalsahab8855', 'ElizavetaVesna', 'nevo01', 'plotogamer', 'thelegendxd8bpq', 'MattMatze', 'ramya254', 'ajedrecista28', 'igraclagan', 'gr33do', 'sefeydn', 'MilanKontic', 'std_priority_queue', 'cool_shine']
Déjà téléchargés : 867
Restants à télécharger : 1023


## 3. Télécharger ou recharger `raw_games_by_player`


In [15]:
downloaded_player_ids = set(list_downloaded_player_ids(RAW_GAMES_BY_PLAYER_DIR))
remaining_player_ids = [p for p in seed_player_ids if p not in downloaded_player_ids]

print(f"Déjà téléchargés : {len(downloaded_player_ids)}")
print(f"Restants à télécharger : {len(remaining_player_ids)}")
print(remaining_player_ids[:10])

Déjà téléchargés : 867
Restants à télécharger : 1023
['vebbev', 'Velociraptor1510', 'starfox_88', 'Fnntnn', 'BENHAIMED-CMPT3', 'imbayog', 'Egordan', 'sunil_goyal', 'nowjohn', 'JohnPion']


In [16]:
failed_players = []

for idx, player_id in enumerate(remaining_player_ids, start=1):
    player_json_path = RAW_GAMES_BY_PLAYER_DIR / f"{player_id}.json"

    # Garde-fou de reprise
    if player_json_path.exists():
        print(f"[SKIP] {player_id} déjà téléchargé")
        continue

    try:
        # 1) Données publiques du joueur
        user_data = client.users.get_public_data(player_id)

        if not user_data or "createdAt" not in user_data:
            raise ValueError("Champ 'createdAt' introuvable dans get_public_data.")

        created_at = user_data["createdAt"]

        # 2) Fenêtre d'observation : 6 mois après création du compte
        since = berserk.utils.to_millis(created_at)
        until = berserk.utils.to_millis(created_at + timedelta(days=183))

        # 3) Export des parties dans cette fenêtre
        games = client.games.export_by_player(
            player_id,
            max=MAX_GAMES_PER_PLAYER,
            since=since,
            until=until,
            perf_type="rapid",
            opening=True,
        )
        games = list(games)

        # 4) Filtrage local complémentaire
        games = [
            g for g in games
            if isinstance(g, dict)
            and g.get("rated") is True
            and (
                str(g.get("speed") or "").lower() == "rapid"
                or str(g.get("perf") or "").lower() == "rapid"
            )
            and str(g.get("status") or "").lower() not in {"created", "started", ""}
        ]

        # 5) Tri du plus ancien au plus récent
        games = sorted(games, key=lambda g: g.get("createdAt"))

        # 6) Sauvegarde immédiate
        save_raw_games_for_one_player(
            player_id=player_id,
            games=games,
            directory=RAW_GAMES_BY_PLAYER_DIR,
        )

        print(
            f"[{idx}/{len(remaining_player_ids)}] "
            f"{player_id} | createdAt={created_at} | {len(games)} parties sauvegardées"
        )

    except Exception as e:
        failed_players.append(player_id)
        print(f"[ERREUR] {player_id}: {e}")

        with FAILED_PLAYERS_PATH.open("a", encoding="utf-8") as f:
            f.write(player_id + "\n")

        # si rate limit, pause plus longue
        if "429" in str(e) or "Too Many Requests" in str(e):
            print("Pause longue après rate limit...")
            time.sleep(30)

    time.sleep(SLEEP_SECONDS)

[ERREUR] vebbev: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Velociraptor1510: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] starfox_88: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Fnntnn: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] BENHAIMED-CMPT3: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] imbayog: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Egordan: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] sunil_goyal: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] nowjohn: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] JohnPion: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] D171016350: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] deVKim: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] unkownahirishere: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Simo01101110: Champ 'createdAt' introuvable dan

In [17]:
raw_games_by_player = load_raw_games_by_player_from_directory(RAW_GAMES_BY_PLAYER_DIR)

print(f"raw_games_by_player reconstruit depuis {RAW_GAMES_BY_PLAYER_DIR}")
print(f"{len(raw_games_by_player)} joueurs chargés depuis le disque")

raw_games_by_player reconstruit depuis ..\data\raw\by_player
1827 joueurs chargés depuis le disque


## 4. Construire ou recharger `player_games_raw`

Ici, on passe du JSON API brut à la table analytique.
La fonction `build_player_games_raw_from_api_dump(...)` doit :

- appeler `build_player_games(games)` pour chaque joueur
- concaténer les DataFrames
- faire `drop_duplicates(subset=['player_id', 'game_id'])`


In [18]:
if USE_EXISTING_PLAYER_GAMES_RAW_JSON and PLAYER_GAMES_RAW_JSON_PATH.exists():
    player_games_raw = load_player_games_raw(PLAYER_GAMES_RAW_JSON_PATH)
    print(f'player_games_raw rechargé depuis {PLAYER_GAMES_RAW_JSON_PATH}')
else:
    player_games_raw = build_player_games_raw_from_api_dump(raw_games_by_player)
    save_player_games_raw(player_games_raw, PLAYER_GAMES_RAW_JSON_PATH)
    print(f'player_games_raw sauvegardé dans {PLAYER_GAMES_RAW_JSON_PATH}')


player_games_raw sauvegardé dans ..\data\raw\player_games_raw.json


In [19]:
print(player_games_raw.shape)
display(player_games_raw.head())


(1112932, 22)


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,suIoXxvP,2025-01-09 19:39:21.207000+00:00,3,a123krowy,A123krowy,denchik1934,Denchik1934,white,0.0,1500,...,resign,1,Queen’s Pawn Systems,D00,Queen's Pawn Game,40,rapid,rapid,True,pool
1,suIoXxvP,2025-01-09 19:39:21.207000+00:00,3,denchik1934,Denchik1934,a123krowy,A123krowy,black,1.0,1505,...,resign,1,Queen’s Pawn Systems,D00,Queen's Pawn Game,40,rapid,rapid,True,pool
2,SgxvW53p,2025-01-09 19:50:32.757000+00:00,3,hlk1985,HLK1985,a123krowy,A123krowy,white,0.5,1108,...,stalemate,1,Queen’s Pawn Systems,D01,"Rapport-Jobava System, with e6",92,rapid,rapid,True,pool
3,SgxvW53p,2025-01-09 19:50:32.757000+00:00,3,a123krowy,A123krowy,hlk1985,HLK1985,black,0.5,1267,...,stalemate,1,Queen’s Pawn Systems,D01,"Rapport-Jobava System, with e6",92,rapid,rapid,True,pool
4,ehUwi7BW,2025-01-09 20:19:41.903000+00:00,3,lifeisachessgame,lifeisachessgame,a123krowy,A123krowy,white,1.0,1357,...,timeout,1,Petrov Defense,C42,Petrov's Defense: Three Knights Game,9,rapid,rapid,True,pool


In [20]:
print('Nb joueurs dans player_games_raw :', player_games_raw['player_id'].nunique())
print('Doublons (player_id, game_id) :', player_games_raw.duplicated(subset=['player_id', 'game_id']).sum())


Nb joueurs dans player_games_raw : 296738
Doublons (player_id, game_id) : 0


## 5. Appliquer la sélection finale


In [21]:
selection_config = PlayerSelectionConfig(
    min_games_in_window=200,
    max_games_in_window=1000,
    burnin_games=20,
    min_initial_elo=1000,
    max_initial_elo=1400,
    observation_window_days=183,
)

player_games_selected = select_player_games(
    player_games_raw,
    config=selection_config,
    verbose=True,
)


[Initial] rows=1,112,932 | players=296,738
[Après filtres bruts] rows=1,112,932 | players=296,738
[Après suppression des 20 premières parties rapid/rated/finished] rows=541,966 | players=1,685
[Après filtre Elo initial [1000, 1400]] rows=337,434 | players=1,150
[Après fenêtre d'observation de 183 jours] rows=335,570 | players=1,150
[Après filtre nombre de parties [200, 1000]] rows=149,251 | players=298

[Résumé final]
Joueurs retenus : 298
Lignes retenues  : 149,251
Nb de parties par joueur :
count    298.000000
mean     500.842282
std      214.950653
min      200.000000
25%      322.250000
50%      470.500000
75%      665.000000
max      998.000000
Name: game_id, dtype: float64

Elo initial dans la fenêtre :
count     298.000000
mean     1160.503356
std       101.031457
min      1000.000000
25%      1083.000000
50%      1149.500000
75%      1243.000000
max      1400.000000
Name: elo_player_before, dtype: float64


## 6. Inspecter les résultats


In [22]:
print('Nb joueurs retenus :', player_games_selected['player_id'].nunique())
print(player_games_selected.groupby('player_id')['game_id'].count().describe())
display(player_games_selected[['player_id', 'elo_player_before']].head())


Nb joueurs retenus : 298
count    298.000000
mean     500.842282
std      214.950653
min      200.000000
25%      322.250000
50%      470.500000
75%      665.000000
max      998.000000
Name: game_id, dtype: float64


,player_id,elo_player_before
0,a123krowy,1071
1,a123krowy,1088
2,a123krowy,1072
3,a123krowy,1053
4,a123krowy,1070


In [23]:
selected_players = sorted(player_games_selected['player_id'].unique())
print(len(selected_players))
print(selected_players[:30])


298
['a123krowy', 'a5_tulatrain-moscow', 'abadan007', 'abasssm', 'abdeerrahim', 'abiramy_ruban', 'abo_omar122', 'aboadel2025', 'aboamer14', 'ad0105', 'adalbertus_szachmat', 'adsiw', 'ahmad189', 'ahmed_9669', 'ahmetsnll', 'ajayguntreddi', 'albina228', 'aleksey5001', 'alessandro949', 'alexanderororo', 'alexey963', 'alexsur99', 'alexviat', 'alicee99', 'aluschirm', 'ameliajuniorchess', 'amir500a', 'amir_klv98', 'anaroque', 'anje36']


In [24]:
games_per_player = player_games_selected.groupby('player_id')['game_id'].count().sort_values(ascending=False)
display(games_per_player.head(143))


player_id
alessandro949        998
erik_osl             995
don81                990
maxicuro             979
mahdiuip             968
                    ... 
rroseselavie         493
rifat-33             491
maksimcapsamun198    491
abasssm              485
bendjedidi_aymen2    483
Name: game_id, Length: 143, dtype: int64

## 7. Sauvegardes complémentaires optionnelles

Si tu veux plus tard, tu pourras ajouter ici :

- sauvegarde de `player_games_selected`
- export de la liste finale des joueurs retenus
- statistiques résumées


In [25]:

# 1. Sauvegarde du dataset final sélectionné
save_player_games_selected(
    player_games_selected,
    PLAYER_GAMES_SELECTED_JSON_PATH,
)
print(f"player_games_selected sauvegardé dans {PLAYER_GAMES_SELECTED_JSON_PATH}")

# 2. Sauvegarde de la liste des joueurs retenus
selected_players = sorted(player_games_selected["player_id"].unique())
SELECTED_PLAYERS_TXT_PATH.parent.mkdir(parents=True, exist_ok=True)
SELECTED_PLAYERS_TXT_PATH.write_text("\n".join(selected_players), encoding="utf-8")
print(f"Liste des joueurs retenus sauvegardée dans {SELECTED_PLAYERS_TXT_PATH}")

# 3. Sauvegarde d'un résumé de sélection
games_per_player = player_games_selected.groupby("player_id")["game_id"].count()
first_games = (
    player_games_selected
    .sort_values(["player_id", "datetime_utc", "game_id"])
    .groupby("player_id", as_index=False)
    .first()
)

selection_summary = {
    "n_players": int(player_games_selected["player_id"].nunique()),
    "n_rows": int(len(player_games_selected)),
    "games_per_player": {
        "count": float(games_per_player.count()),
        "mean": float(games_per_player.mean()),
        "std": float(games_per_player.std()),
        "min": float(games_per_player.min()),
        "25%": float(games_per_player.quantile(0.25)),
        "50%": float(games_per_player.quantile(0.50)),
        "75%": float(games_per_player.quantile(0.75)),
        "max": float(games_per_player.max()),
    },
    "initial_elo_in_window": {
        "count": float(first_games["elo_player_before"].count()),
        "mean": float(first_games["elo_player_before"].mean()),
        "std": float(first_games["elo_player_before"].std()),
        "min": float(first_games["elo_player_before"].min()),
        "25%": float(first_games["elo_player_before"].quantile(0.25)),
        "50%": float(first_games["elo_player_before"].quantile(0.50)),
        "75%": float(first_games["elo_player_before"].quantile(0.75)),
        "max": float(first_games["elo_player_before"].max()),
    },
}

SELECTION_SUMMARY_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with SELECTION_SUMMARY_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(selection_summary, f, ensure_ascii=False, indent=2)

print(f"Résumé de sélection sauvegardé dans {SELECTION_SUMMARY_JSON_PATH}")



selection_config_dict = {
    "min_games_after_burnin": 200,
    "max_games": 1000,
    "burnin_games": 20,
    "min_initial_elo": 1000,
    "max_initial_elo": 1400,
    "observation_window_days": 183,
}


with (SELECTION_CONFIG_JSON_PATH).open("w", encoding="utf-8") as f:
    json.dump(selection_config_dict, f, indent=2)

player_games_selected sauvegardé dans ..\data\selected\player_games_selected.json
Liste des joueurs retenus sauvegardée dans ..\data\selected\selected_players.txt
Résumé de sélection sauvegardé dans ..\data\selected\selection_summary.json


In [26]:
assert player_games_selected["player_id"].nunique() > 0
assert not player_games_selected.empty
assert player_games_selected.duplicated(subset=["player_id", "game_id"]).sum() == 0

# 8. Création des parties enrichies

In [38]:
from src.features.game_features import build_game_features

games_final = build_game_features(player_games_selected)

print("Shape games_final :", games_final.shape)
display(games_final.head())

Shape games_final : (149251, 45)


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,session_delay,session_discrete_delay,day_date_utc,day_n_games,day_score,day_position,week_id,week_n_games,week_score,week_position
0,pP77HkPM,2025-01-18 09:07:25.062000+00:00,5,a123krowy,A123krowy,himanshi2015,Himanshi2015,white,1.0,1071,...,NaN,NaN,2025-01-18,1,1.00,0,2025-W03,11,0.409091,0
1,6HwUpUTM,2025-01-19 07:55:33.921000+00:00,6,a123krowy,A123krowy,toni66,toni66,black,0.0,1088,...,82088.859,B,2025-01-19,10,0.35,0,2025-W03,11,0.409091,1
2,HESDVI1Z,2025-01-19 08:13:55.722000+00:00,6,a123krowy,A123krowy,gabrielc400,gabrielc400,black,0.0,1072,...,82088.859,B,2025-01-19,10,0.35,1,2025-W03,11,0.409091,2
3,n7s1jj71,2025-01-19 08:18:30.434000+00:00,6,a123krowy,A123krowy,neggm1,neggm1,black,1.0,1053,...,82088.859,B,2025-01-19,10,0.35,2,2025-W03,11,0.409091,3
4,DIiwuSXs,2025-01-19 09:29:37.424000+00:00,6,a123krowy,A123krowy,huseyinaltan,HUSEYINALTAN,white,0.0,1070,...,4266.990,A,2025-01-19,10,0.35,3,2025-W03,11,0.409091,4


## 8.1 Quelques tests

In [39]:
assert len(games_final) == len(player_games_selected), \
    f"Perte de lignes : {len(player_games_selected)} → {len(games_final)}"

duplicates = games_final.duplicated(subset=["player_id", "game_id"]).sum()
assert duplicates == 0, f"Doublons détectés : {duplicates}"

session_stats = games_final.groupby("player_id")["session_id"].nunique()

print(session_stats.describe())

count    298.000000
mean     185.419463
std      105.018398
min       21.000000
25%      110.000000
50%      164.500000
75%      248.000000
max      551.000000
Name: session_id, dtype: float64


In [29]:
delay_stats = games_final["delay_previous_game"].describe()
print(delay_stats)

count    1.489530e+05
mean     2.499709e+04
std      1.442821e+05
min      3.091000e+00
25%      4.665940e+02
50%      1.128560e+03
75%      1.285118e+04
max      1.442312e+07
Name: delay_previous_game, dtype: float64


In [41]:
display(
    games_final[
        ["player_id", "game_id", "streak_before", "streak_after"]
    ].head(20)
)

,player_id,game_id,streak_before,streak_after
0,a123krowy,pP77HkPM,NaN,1
1,a123krowy,6HwUpUTM,1.0,-1
2,a123krowy,HESDVI1Z,-1.0,-2
3,a123krowy,n7s1jj71,-2.0,1
4,a123krowy,DIiwuSXs,1.0,-1
5,a123krowy,rkFHTF34,-1.0,1
6,a123krowy,wIaJVpXJ,1.0,-1
7,a123krowy,T7dz4wQu,-1.0,-2
8,a123krowy,AQExDB2f,-2.0,0
9,a123krowy,iS5YGelz,0.0,-1


In [43]:
display(
    games_final[["day_n_games", "week_n_games"]].describe()
)

,day_n_games,week_n_games
count,149251.000000,149251.000000
mean,11.534965,49.945964
std,9.675731,36.830471
min,1.000000,1.000000
25%,5.000000,25.000000
50%,9.000000,41.000000
75%,15.000000,64.000000
max,76.000000,293.000000


## 8.2 Sauvegarde des parties

In [ ]:


GAMES_FINAL_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

save_player_games_selected(
    games_final,
    GAMES_FINAL_JSON_PATH,
)

print(f"games_final sauvegardé dans {GAMES_FINAL_JSON_PATH}")



games_final sauvegardé dans ..\data\enriched\games_final.json


# 9. Création des données niveau joueur

In [8]:
games_final = load_player_games_selected(GAMES_FINAL_JSON_PATH)

games_final.head(20)

,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,session_delay,session_discrete_delay,day_date_utc,day_n_games,day_score,day_position,week_id,week_n_games,week_score,week_position
0,pP77HkPM,2025-01-18 09:07:25.062000+00:00,5,a123krowy,A123krowy,himanshi2015,Himanshi2015,white,1.0,1071,...,NaN,NaN,2025-01-18,1,1.00,0,2025-W03,11,0.409091,0
1,6HwUpUTM,2025-01-19 07:55:33.921000+00:00,6,a123krowy,A123krowy,toni66,toni66,black,0.0,1088,...,82088.859,B,2025-01-19,10,0.35,0,2025-W03,11,0.409091,1
2,HESDVI1Z,2025-01-19 08:13:55.722000+00:00,6,a123krowy,A123krowy,gabrielc400,gabrielc400,black,0.0,1072,...,82088.859,B,2025-01-19,10,0.35,1,2025-W03,11,0.409091,2
3,n7s1jj71,2025-01-19 08:18:30.434000+00:00,6,a123krowy,A123krowy,neggm1,neggm1,black,1.0,1053,...,82088.859,B,2025-01-19,10,0.35,2,2025-W03,11,0.409091,3
4,DIiwuSXs,2025-01-19 09:29:37.424000+00:00,6,a123krowy,A123krowy,huseyinaltan,HUSEYINALTAN,white,0.0,1070,...,4266.990,A,2025-01-19,10,0.35,3,2025-W03,11,0.409091,4
5,rkFHTF34,2025-01-19 10:26:22.007000+00:00,6,a123krowy,A123krowy,rahith_jalan,Rahith_Jalan,black,1.0,1057,...,4266.990,A,2025-01-19,10,0.35,4,2025-W03,11,0.409091,5
6,wIaJVpXJ,2025-01-19 11:09:06.883000+00:00,6,a123krowy,A123krowy,diman_zzz,Diman_ZzZ,white,0.0,1071,...,4266.990,A,2025-01-19,10,0.35,5,2025-W03,11,0.409091,6
7,T7dz4wQu,2025-01-19 18:27:20.702000+00:00,6,a123krowy,A123krowy,rajab2016,Rajab2016,white,0.0,1058,...,26293.819,B,2025-01-19,10,0.35,6,2025-W03,11,0.409091,7
8,AQExDB2f,2025-01-19 18:36:00.432000+00:00,6,a123krowy,A123krowy,josemanuelmm,JoseManuelMM,black,0.5,1044,...,26293.819,B,2025-01-19,10,0.35,7,2025-W03,11,0.409091,8
9,iS5YGelz,2025-01-19 19:04:10.214000+00:00,6,a123krowy,A123krowy,simonebruscino,SimoneBruscino,white,0.0,1042,...,26293.819,B,2025-01-19,10,0.35,8,2025-W03,11,0.409091,9


In [ ]:
from src.features.player_features import build_player_features
from src.labels.progression import build_progression_labels

# 1. Features joueur
player_features = build_player_features(games_final)

# 2. Labels de progression
progression_labels = build_progression_labels(games_final)

# 3. Merge final
final_dataset = player_features.merge(
    progression_labels,
    on="player_id",
    how="inner"
)

player_features : (298, 30)
progression_labels : (298, 5)
final_dataset : (298, 34)


,player_id,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration,abandon_rate,time_loss_rate,draw_ratio,score_when_winstreak,...,entropy_sessions_interval,mean_games_per_session,cv_games_per_session,increment_game_ratio,weekday_bias,color_bias,session_length_performance_slope,within_session_performance_slope,games_per_day_performance_slope,games_per_week_performance_slope
0,a123krowy,41.511269,Queen’s Pawn Systems,King’s Pawn Games / Bishop’s Opening / Vienna,1.498863,0.444997,0.386714,0.021352,0.035587,0.536313,...,1.218249,5.078313,0.909771,0.996441,-0.028967,0.118548,0.006886,0.023772,0.003289,0.001328
1,a5_tulatrain-moscow,47.283871,King’s Gambit,Sicilian,1.693547,0.498364,0.390323,0.038710,0.027419,0.347059,...,1.141608,2.695652,0.912212,0.085484,-0.053779,0.079765,0.023228,-0.071883,0.009776,0.001818
2,abadan007,71.824885,Italian Game & Two Knights Defense,Italian Game & Two Knights Defense,2.434890,0.225493,0.036866,0.018433,0.032258,0.480000,...,1.313033,1.486301,0.592560,1.000000,-0.092144,-0.046084,-0.125320,0.307955,-0.030455,-0.001012
3,abasssm,45.824742,King’s Pawn Games / Bishop’s Opening / Vienna,King’s Pawn Games / Bishop’s Opening / Vienna,2.261723,0.238830,0.274227,0.030928,0.014433,0.500000,...,1.294544,3.368056,0.827018,0.000000,-0.002235,0.102959,0.004610,0.026876,0.001997,0.000547
4,abdeerrahim,63.962672,Italian Game & Two Knights Defense,Modern / Robatsch / Pirc,2.108971,0.359022,0.204322,0.041257,0.023576,0.568000,...,1.078820,2.203463,0.811527,0.009823,0.003827,0.017504,0.011173,0.011297,0.005349,-0.001877


,player_id,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
0,a123krowy,-376.0,-0.446556,0.003464,0.297313
1,a5_tulatrain-moscow,-133.0,-0.214863,-0.241720,-1.013004
2,abadan007,39.0,0.180556,0.151171,0.237955
3,abasssm,21.0,0.043388,0.166094,0.448836
4,abdeerrahim,136.0,0.267717,0.268417,0.892955


,player_id,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration,abandon_rate,time_loss_rate,draw_ratio,score_when_winstreak,...,weekday_bias,color_bias,session_length_performance_slope,within_session_performance_slope,games_per_day_performance_slope,games_per_week_performance_slope,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
0,a123krowy,41.511269,Queen’s Pawn Systems,King’s Pawn Games / Bishop’s Opening / Vienna,1.498863,0.444997,0.386714,0.021352,0.035587,0.536313,...,-0.028967,0.118548,0.006886,0.023772,0.003289,0.001328,-376.0,-0.446556,0.003464,0.297313
1,a5_tulatrain-moscow,47.283871,King’s Gambit,Sicilian,1.693547,0.498364,0.390323,0.038710,0.027419,0.347059,...,-0.053779,0.079765,0.023228,-0.071883,0.009776,0.001818,-133.0,-0.214863,-0.241720,-1.013004
2,abadan007,71.824885,Italian Game & Two Knights Defense,Italian Game & Two Knights Defense,2.434890,0.225493,0.036866,0.018433,0.032258,0.480000,...,-0.092144,-0.046084,-0.125320,0.307955,-0.030455,-0.001012,39.0,0.180556,0.151171,0.237955
3,abasssm,45.824742,King’s Pawn Games / Bishop’s Opening / Vienna,King’s Pawn Games / Bishop’s Opening / Vienna,2.261723,0.238830,0.274227,0.030928,0.014433,0.500000,...,-0.002235,0.102959,0.004610,0.026876,0.001997,0.000547,21.0,0.043388,0.166094,0.448836
4,abdeerrahim,63.962672,Italian Game & Two Knights Defense,Modern / Robatsch / Pirc,2.108971,0.359022,0.204322,0.041257,0.023576,0.568000,...,0.003827,0.017504,0.011173,0.011297,0.005349,-0.001877,136.0,0.267717,0.268417,0.892955


In [12]:

print("player_features :", player_features.shape)
print("progression_labels :", progression_labels.shape)
print("final_dataset :", final_dataset.shape)

display(player_features.head())
display(progression_labels.head())
display(final_dataset.head())

player_features : (298, 30)
progression_labels : (298, 5)
final_dataset : (298, 34)


,player_id,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration,abandon_rate,time_loss_rate,draw_ratio,score_when_winstreak,...,entropy_sessions_interval,mean_games_per_session,cv_games_per_session,increment_game_ratio,weekday_bias,color_bias,session_length_performance_slope,within_session_performance_slope,games_per_day_performance_slope,games_per_week_performance_slope
0,a123krowy,41.511269,Queen’s Pawn Systems,King’s Pawn Games / Bishop’s Opening / Vienna,1.498863,0.444997,0.386714,0.021352,0.035587,0.536313,...,1.218249,5.078313,0.909771,0.996441,-0.028967,0.118548,0.006886,0.023772,0.003289,0.001328
1,a5_tulatrain-moscow,47.283871,King’s Gambit,Sicilian,1.693547,0.498364,0.390323,0.038710,0.027419,0.347059,...,1.141608,2.695652,0.912212,0.085484,-0.053779,0.079765,0.023228,-0.071883,0.009776,0.001818
2,abadan007,71.824885,Italian Game & Two Knights Defense,Italian Game & Two Knights Defense,2.434890,0.225493,0.036866,0.018433,0.032258,0.480000,...,1.313033,1.486301,0.592560,1.000000,-0.092144,-0.046084,-0.125320,0.307955,-0.030455,-0.001012
3,abasssm,45.824742,King’s Pawn Games / Bishop’s Opening / Vienna,King’s Pawn Games / Bishop’s Opening / Vienna,2.261723,0.238830,0.274227,0.030928,0.014433,0.500000,...,1.294544,3.368056,0.827018,0.000000,-0.002235,0.102959,0.004610,0.026876,0.001997,0.000547
4,abdeerrahim,63.962672,Italian Game & Two Knights Defense,Modern / Robatsch / Pirc,2.108971,0.359022,0.204322,0.041257,0.023576,0.568000,...,1.078820,2.203463,0.811527,0.009823,0.003827,0.017504,0.011173,0.011297,0.005349,-0.001877


,player_id,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
0,a123krowy,-376.0,-0.446556,0.003464,0.297313
1,a5_tulatrain-moscow,-133.0,-0.214863,-0.241720,-1.013004
2,abadan007,39.0,0.180556,0.151171,0.237955
3,abasssm,21.0,0.043388,0.166094,0.448836
4,abdeerrahim,136.0,0.267717,0.268417,0.892955


,player_id,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration,abandon_rate,time_loss_rate,draw_ratio,score_when_winstreak,...,weekday_bias,color_bias,session_length_performance_slope,within_session_performance_slope,games_per_day_performance_slope,games_per_week_performance_slope,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
0,a123krowy,41.511269,Queen’s Pawn Systems,King’s Pawn Games / Bishop’s Opening / Vienna,1.498863,0.444997,0.386714,0.021352,0.035587,0.536313,...,-0.028967,0.118548,0.006886,0.023772,0.003289,0.001328,-376.0,-0.446556,0.003464,0.297313
1,a5_tulatrain-moscow,47.283871,King’s Gambit,Sicilian,1.693547,0.498364,0.390323,0.038710,0.027419,0.347059,...,-0.053779,0.079765,0.023228,-0.071883,0.009776,0.001818,-133.0,-0.214863,-0.241720,-1.013004
2,abadan007,71.824885,Italian Game & Two Knights Defense,Italian Game & Two Knights Defense,2.434890,0.225493,0.036866,0.018433,0.032258,0.480000,...,-0.092144,-0.046084,-0.125320,0.307955,-0.030455,-0.001012,39.0,0.180556,0.151171,0.237955
3,abasssm,45.824742,King’s Pawn Games / Bishop’s Opening / Vienna,King’s Pawn Games / Bishop’s Opening / Vienna,2.261723,0.238830,0.274227,0.030928,0.014433,0.500000,...,-0.002235,0.102959,0.004610,0.026876,0.001997,0.000547,21.0,0.043388,0.166094,0.448836
4,abdeerrahim,63.962672,Italian Game & Two Knights Defense,Modern / Robatsch / Pirc,2.108971,0.359022,0.204322,0.041257,0.023576,0.568000,...,0.003827,0.017504,0.011173,0.011297,0.005349,-0.001877,136.0,0.267717,0.268417,0.892955


## 9.1 Quelques tests

In [13]:
assert player_features["player_id"].nunique() == len(player_features), \
    "player_features ne contient pas exactement une ligne par joueur."

assert progression_labels["player_id"].nunique() == len(progression_labels), \
    "progression_labels ne contient pas exactement une ligne par joueur."

In [14]:
n_final = final_dataset["player_id"].nunique()
n_feat = player_features["player_id"].nunique()
n_prog = progression_labels["player_id"].nunique()

print("Joueurs final_dataset       :", n_final)
print("Joueurs player_features    :", n_feat)
print("Joueurs progression_labels :", n_prog)

assert n_final == n_feat == n_prog, \
    "Les populations de joueurs ne correspondent pas entre les DataFrames."

Joueurs final_dataset       : 298
Joueurs player_features    : 298
Joueurs progression_labels : 298


In [15]:
assert final_dataset["player_id"].nunique() == len(final_dataset), \
    "final_dataset ne contient pas exactement une ligne par joueur."

print("Nb colonnes final_dataset :", final_dataset.shape[1])
print("Colonnes :")
print(final_dataset.columns.tolist())

Nb colonnes final_dataset : 34
Colonnes :
['player_id', 'mean_ply_count', 'main_opening_white', 'main_opening_black', 'opening_diversity', 'opening_concentration', 'abandon_rate', 'time_loss_rate', 'draw_ratio', 'score_when_winstreak', 'score_when_losestreak', 'streak_resilience_bias', 'abandon_rate_when_losestreak', 'time_loss_rate_when_losestreak', 'delay_ratio_when_winstreak', 'delay_ratio_when_losestreak', 'cv_games_interval', 'cv_games_per_day', 'cv_games_per_week', 'cv_sessions_interval', 'entropy_sessions_interval', 'mean_games_per_session', 'cv_games_per_session', 'increment_game_ratio', 'weekday_bias', 'color_bias', 'session_length_performance_slope', 'within_session_performance_slope', 'games_per_day_performance_slope', 'games_per_week_performance_slope', 'elo_gain', 'elo_gain_per_game', 'elo_slope_per_game', 'elo_slope_per_day']


In [16]:
assert final_dataset.duplicated(subset=["player_id"]).sum() == 0, \
    "Doublons détectés sur player_id dans final_dataset."

In [17]:
nan_summary = final_dataset.isna().mean().sort_values(ascending=False)
display(nan_summary[nan_summary > 0])

Series([], dtype: float64)

## 9.2 Sauvegarde du dataset final

In [19]:
FINAL_DATASET_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

save_player_games_selected(
    final_dataset,
    FINAL_DATASET_JSON_PATH,
)

print(f"final_dataset sauvegardé dans {FINAL_DATASET_JSON_PATH}")

final_dataset sauvegardé dans ..\data\final\final_dataset.json
